# Reduktion af sensorredundans på produktionslinjen med PROC GVARCLUS

## Resumé

En produktionslinje med flere zoner strømmer snesevis af korrelerede sensoraflæsninger, hvoraf mange bærer det samme underliggende signal. Denne notebook bruger **PROC GVARCLUS** (variabelklyngeanalyse) til at gruppere de 13 procesensorer i fire adskilte klynger, og aflæser derefter hver klynges R-kvadrat-værdier for at vælge én repræsentativ måler pr. klynge — hvilket reducerer SPC-overvågningens omfang uden at miste procesinformation. Tre af de fire klynger følger pænt de fysiske zoner (gennemsnitlig R-kvadrat 0,92, 0,93 og 0,96); den fjerde er en enkeltkanals-klynge, som proceduren udskilte for sig selv, hvilket vi undersøger nærmere i stedet for at overse det.

## Datakilder

Alle data genereres direkte med `call streaminit(20260531)` og `rand()` — ingen eksterne eller netværksbaserede inputs.

| Datasæt | Rækker | Nøglevariable | Beskrivelse |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Syntetiske aflæsninger fra en produktionslinje med 3 zoner. Sensorer inden for en zone deler en latent drivkraft (høj korrelation); tværgående målere (temperaturer koblet til zone 1, tryk til zone 2, vibration til zone 3) er tilføjet for at skabe realistisk redundans. DATA-step-løkken anmoder om 300 cyklusser, men dette evalueringsmiljø kører i ulicenseret tilstand og bevarer kun de første 100 observationer — nok til at genskabe klyngestrukturen tydeligt. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Én række pr. inputsensor: den klynge, den blev tildelt, og dens R-kvadrat med den klynges komponent. Produceret af `OUTPUT OUT=`-sætningen. |

# Reduktion af sensorredundans på produktionslinjen med PROC GVARCLUS

På en instrumenteret produktionslinje er det almindeligt at logge snesevis af sensorer, der måler overlappende fysiske fænomener — flere termoelementer i én zone, redundante tryktransducere, vibrationsprober der følger samme aksel. At sende hver eneste kanal ind i et kontroldiagram eller en downstream-model spilder overvågningsindsats og øger multikollinearitet.

**Variabelklyngeanalyse** løser dette direkte. `PROC GVARCLUS` opdeler de numeriske variable i *adskilte* klynger, hvor hver klynge opsummeres af den første principale komponent blandt sine medlemmer. Sensorer, der bevæger sig sammen, havner i samme klynge; ingeniøren kan derefter beholde én repræsentativ kanal pr. klynge.

I denne notebook:

1. Syntetiserer vi aflæsninger fra en linje med 3 zoner med bevidst redundante sensorer (løkken anmoder om 300 cyklusser; 100 bevares her).
2. Klynger vi de 13 kanaler med `PROC GVARCLUS` ved `MAXCLUSTERS=4`.
3. Fanger vi klyngetildelingerne i et outputdatasæt og opsummerer dem.
4. Fortolker vi R-kvadrat-værdierne for at vælge én måler pr. klynge til løbende SPC.

## Trin 1 — Generér syntetiske sensordata

Vi simulerer tre proceszoner, hver med en skjult drivkraft (`zone1_a`, `zone2_a`, `zone3_a`). De resterende kanaler er støjfyldte lineære funktioner af deres zones drivkraft, så de er stærkt korrelerede inden for en zone, men næsten uafhængige på tværs af zoner. Vi kobler også indløbs-/udløbstemperaturerne til zone 1 og de to tryktransducere til zone 2, hvilket efterligner den tværinstrumentelle redundans, man ser på virkelige linjer. Et fast frø gør kørslen reproducerbar. Løkken anmoder om 300 cyklusser; i ulicenseret tilstand bevarer motoren de første 100 observationer, hvilket NOTE-linjen nedenfor rapporterer.

In [1]:
data process_sensors;
    CALL streaminit(20260531);
    GØR cycle = 1 TIL 300;
        /* Zone 1: latent drivkraft plus to redundante prober */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latent drivkraft plus to redundante prober */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latent drivkraft plus én redundant probe */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Tværinstrumentelle kanaler koblet til eksisterende drivkræfter */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        UDDATA;
    SLUT;
    FJERN cycle;
KØR;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds


## Trin 2 — Klyng sensorerne

Vi kalder `PROC GVARCLUS` på alle 13 kanaler. `VAR`-sætningen angiver de numeriske sensorer, der skal klynges. Vi begrænser opdelingen til fire klynger med `MAXCLUSTERS=4` (vi forventer omtrent én klynge pr. fysisk zone, med lidt slør). `ODS GRAPHICS ON` med `PLOTS=ALL` aktiverer variabel-klynge-dendrogrammet; motoren skriver denne SVG til arbejdsmappen i stedet for at vise den direkte, så strukturen, vi aflæser nedenfor, kommer fra den udskrevne tabel **Variable Cluster Assignments** og eigenværditabellen pr. klynge.

`OUTPUT OUT=`-sætningen skriver variabel-til-klynge-tildelingerne — sammen med hver variabels R-kvadrat mod sin egen klynge — til et datasæt, vi kan programmere videre på senere.

In [2]:
ODS GRAPHICS ON;

PROCEDURE gvarclus data=process_sensors maxclusters=4 PLOTS=ALL;
    VARIABEL zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    UDDATA out=clusters;
KØR;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Trin 3 — Kør igen med SUMMARY-tilvalget

At køre proceduren en anden gang med tilvalget `SUMMARY` bevarer den samme opdeling. `PLOTS=NONE` slår grafik fra i denne kørsel. I dette miljø er den udskrevne rapport identisk med trin 2 — den samme 13-rækkers tildelingstabel, de samme fire klynger og den samme eigenværdioversigt — så denne celle demonstrerer primært, at tilvalgene `SUMMARY` og `PLOTS=NONE` parses og kører korrekt; den forventes ikke at tilføje nye tal.

In [3]:
PROCEDURE gvarclus data=process_sensors maxclusters=4 summary PLOTS=none;
    VARIABEL zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
KØR;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Trin 4 — Undersøg outputdatasættet

Datasættet `clusters` fra `OUTPUT OUT=` har én række pr. sensor med dens tildelte `Cluster` og `RSq_Own` (den kvadrerede korrelation mellem sensoren og dens klyngekomponent). En høj `RSq_Own` betyder, at sensoren er godt repræsenteret af sin klynge — en ideel kandidat til at *droppe* til fordel for klyngens repræsentant. Vi sorterer efter klynge og derefter efter R-kvadrat, så den mest repræsentative sensor i hver klynge ligger øverst i sin gruppe.

In [4]:
PROCEDURE SORTER data=clusters out=clusters_ranked;
    EFTER CLUSTER FALDENDE RSq_Own;
KØR;

PROCEDURE UDSKRIV data=clusters_ranked MÆRKAT;
    VARIABEL Variable CLUSTER RSq_Own;
    MÆRKAT Variable = "Sensorkanal"
          CLUSTER  = "Klynge"
          RSq_Own  = "R-kvadrat med egen klynge";
KØR;


  Obs  Sensorkanal  Klynge  R-kvadrat med egen klynge
-----  -----------  ------  -------------------------
    1  ZONE1_A           1                    0.96867
    2  ZONE1_B           1                     0.9189
    3  TEMP_IN           1                   0.903394
    4  TEMP_OUT          1                   0.889519
    5  ZONE2_A           2                    0.98364
    6  ZONE2_B           2                   0.946708
    7  PRESSURE_A        2                   0.929356
    8  PRESSURE_B        2                   0.920915
    9  ZONE2_C           2                   0.892405
   10  ZONE3_A           3                   0.977237
   11  VIBRATION         3                    0.95916
   12  ZONE3_B           3                   0.949054
   13  ZONE1_C           4                          1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Fortolkning af resultaterne

Opdelingen genskaber det meste af linjens fysiske struktur, med ét ærligt forbehold:

- **Klynge 1** samler `zone1_a` (R²=0,969), `zone1_b` (0,919) samt indløbs-/udløbstemperaturerne `temp_in` (0,903) og `temp_out` (0,890) — fire af de fem kanaler drevet af zone-1-signalet. Gennemsnitlig R-kvadrat 0,920.
- **Klynge 2** samler alle tre zone-2-kanaler `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) sammen med de to tryktransducere `pressure_a` (0,929) og `pressure_b` (0,921), som vi koblede til zone-2-drivkraften. Gennemsnitlig R-kvadrat 0,935.
- **Klynge 3** samler `zone3_a` (0,977), `zone3_b` (0,949) og `vibration` (0,959). Gennemsnitlig R-kvadrat 0,962.
- **Klynge 4** er en enkelt kanal: `zone1_c`, den tredje zone-1-probe, blev udskilt for sig selv med R²=1,000 (en singleton er trivielt set perfekt forklaret af sig selv). Selvom den deler zone-1-drivkraften, vurderede proceduren ved denne stikprøvestørrelse, at `zone1_c` var distinkt nok fra `zone1_a`/temperatur-gruppen til at berettige sin egen klynge i stedet for at blive lagt sammen med Klynge 1.

De tre flerkanals-klynger har hver en gennemsnitlig R-kvadrat over **0,90**, hvilket bekræfter, at én komponent forklarer størstedelen af variationen blandt deres medlemmer — netop den redundans, vi ønsker at samle. Den enlige outlier — at `zone1_c` havner i sin egen klynge i stedet for sammen med resten af zone 1 — er en nyttig påmindelse om, at variabelklyngeanalyse er datadrevet: med flere cyklusser eller et højere `MAXCLUSTERS`-budget kan grænsen flytte sig, så opdelingen er et udgangspunkt for ingeniørmæssig vurdering, ikke en fast sandhed.

**Handling for linjen.** Inden for hver flerkanals-klynge beholdes den sensor med højeste `RSq_Own` (den kanal, der er mest repræsentativ for sin klynge), mens resten pensioneres eller nedprioriteres fra rutinemæssig SPC-diagramføring — for eksempel `zone2_a` for Klynge 2 og `zone3_a` for Klynge 3. Behandl `zone1_c`-singletonen som et flag til undersøgelse snarere end en automatisk beholdelse: bekræft, om den bærer genuint distinkt information, eller om det er en klyngeartefakt, før du beslutter at overvåge den separat. Selv med denne ene kanal holdt til side samler dette 13 overvågede kanaler til en fireskanals overvågningsplan, hvilket reducerer alarmstøj og multikollinearitet, samtidig med at linjens informationsindhold bevares.